# 🎧 Shadowing kutubxonasini qurish (Google Colab)

Bu notebook quyidagilarni bajaradi:
1. GitHub repongizni shu Colab muhitiga klonlaydi
2. Kerakli kutubxonalarni (`yt-dlp`, `pydub`, `ffmpeg`) o'rnatadi
3. Siz kiritgan ochiq manba (VOA/BBC/LibriVox) havolalaridan audio segmentlar kutubxonasini quradi
4. Natijani (`shadowing_data/` papkasi) qaytarib GitHub repongizga yuklaydi

**Talab qilinadigan narsalar:**
- GitHub repo havolangiz (masalan `https://github.com/jergashev-1/cefr-bot`)
- GitHub **Personal Access Token** (push qilish uchun) — https://github.com/settings/tokens sahifasidan `repo` huquqi bilan yarating
- `GROQ_API_KEY` — botingizda ishlatilayotgan Groq API kaliti (Whisper transkripsiya uchun)

Repoda `shadowing.py`, `shadowing_library.py`, `build_shadowing_library.py` fayllari va `shadowing_data/` papkasi allaqachon bo'lishi kerak (avval GitHub veb-saytidan yuklab qo'ygan bo'lishingiz kerak).

In [ ]:
#@title 1) Sozlamalarni kiriting { display-mode: "form" }
GITHUB_REPO_URL = "https://github.com/USERNAME/cefr-bot"  #@param {type:"string"}
GITHUB_TOKEN = ""  #@param {type:"string"}
GROQ_API_KEY = ""  #@param {type:"string"}
GIT_USER_NAME = "Colab Bot Builder"  #@param {type:"string"}
GIT_USER_EMAIL = "colab@example.com"  #@param {type:"string"}

import os
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

assert GITHUB_REPO_URL.startswith("https://github.com/"), "To'g'ri GitHub havolasini kiriting"
assert GITHUB_TOKEN, "GitHub Personal Access Token kiritilishi shart (push qilish uchun)"
assert GROQ_API_KEY, "GROQ_API_KEY kiritilishi shart (Whisper transkripsiya uchun)"
print("✅ Sozlamalar qabul qilindi")

In [ ]:
#@title 2) Kerakli dasturlarni o'rnatish
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip install -q yt-dlp pydub groq python-dotenv
print("✅ O'rnatildi")

In [ ]:
#@title 3) Repongizni klonlash
import re

repo_name = GITHUB_REPO_URL.rstrip("/").split("/")[-1]
auth_url = re.sub(
    r"^https://",
    f"https://{GITHUB_TOKEN}@",
    GITHUB_REPO_URL,
)

!rm -rf {repo_name}
!git clone {auth_url}
%cd {repo_name}
!git config user.name "{GIT_USER_NAME}"
!git config user.email "{GIT_USER_EMAIL}"
print("✅ Repo klonlandi:", repo_name)

In [ ]:
#@title 4) Kerakli fayllar mavjudligini tekshirish
import os

required = ["bot.py", "transcriber.py", "shadowing.py", "shadowing_library.py", "build_shadowing_library.py"]
missing = [f for f in required if not os.path.exists(f)]

if missing:
    print("❌ Quyidagi fayllar repoda topilmadi:", missing)
    print("Avval GitHub veb-saytidan shadowing fayllarini repoga yuklab, keyin qayta urinib ko'ring.")
else:
    print("✅ Barcha kerakli fayllar mavjud")

## 5) Manba havolalarini kiriting

Quyidagi katakchada `SOURCES` ro'yxatiga o'zingiz tanlagan **ochiq manba** (VOA Learning English, BBC Learning English, LibriVox va h.k.) videolarining havolalarini qo'shing. Har biri **alohida video/epizod** havolasi bo'lishi kerak (kanal yoki dastur sahifasi emas).

In [ ]:
#@title Manbalarni shu yerga yozing
import json

SOURCES = [
    {
        "name": "VOA Learning English - Namuna dars",
        "url": "https://www.youtube.com/watch?v=XXXXXXXXXXX",
        "level": "easy",
    },
    {
        "name": "BBC 6 Minute English - Namuna epizod",
        "url": "https://www.youtube.com/watch?v=YYYYYYYYYYY",
        "level": "medium",
    },
    # Kerak bo'lsa yana qo'shing:
    # {"name": "...", "url": "...", "level": "easy" | "medium" | "hard"},
]

os.makedirs("shadowing_data", exist_ok=True)
with open("shadowing_data/sources.json", "w", encoding="utf-8") as f:
    json.dump(SOURCES, f, ensure_ascii=False, indent=2)

print(f"✅ {len(SOURCES)} ta manba yozildi -> shadowing_data/sources.json")

In [ ]:
#@title 6) Kutubxonani qurish (bu bir necha daqiqa vaqt olishi mumkin)
!python build_shadowing_library.py

In [ ]:
#@title 7) Natijani tekshirish
import json

with open("shadowing_data/manifest.json", encoding="utf-8") as f:
    manifest = json.load(f)

print(f"Jami segmentlar soni: {len(manifest)}\n")
for item in manifest[:5]:
    print(f"- [{item['level']}] {item['duration_sec']}s | {item['text'][:60]}...")

if len(manifest) > 5:
    print(f"... va yana {len(manifest) - 5} ta segment")

## 8) Natijani GitHub'ga qaytarib yuklash

Agar yuqoridagi natija sizga yoqsa (segmentlar soni yetarli, matnlar to'g'ri ko'rinsa), quyidagi katakchani ishga tushirib, `shadowing_data/` papkasini repongizga push qiling.

In [ ]:
#@title Git orqali push qilish
!git add shadowing_data/
!git commit -m "Shadowing kutubxonasi: yangi audio segmentlar qo'shildi"
!git push origin main
print("✅ Push qilindi. Render/serveringiz avtomatik qayta deploy bo'lishi kerak (agar shunday sozlangan bo'lsa).")

### ℹ️ Eslatmalar

- Agar `git push` xatolik bersa ("failed to push"), avval `!git pull origin main --rebase` ni ishga tushirib, keyin qayta push qilib ko'ring.
- Agar asosiy branch nomi `main` emas, `master` bo'lsa, oxirgi katakchada `main` o'rniga `master` yozing.
- Kutubxonani kengaytirish uchun shunchaki yuqoriga qaytib, yangi manbalar bilan 5–8 qadamlarni qayta bajaring — bu safar oldingi segmentlar o'chirilmaydi, ustiga qo'shiladi.
- GROQ_API_KEY va GitHub tokeningizni hech kim bilan baham ko'rmang; bu notebook nusxasini ulashsangiz, avval shu qatorlarni tozalang.